# 01 — Chatting with a Local LLM via Ollama

**Ollama** runs large language models (LLMs) entirely on your own machine — no internet required, no API costs, full privacy.

## Before you start

1. **Install Ollama**: https://ollama.com  
2. **Start Ollama** (if it's not running as a background service):  
   Open a terminal and run: `ollama serve`
3. **Pull a model**:  
   `ollama pull llama3.2:3b` — good balance of speed and quality (~2GB)  
   `ollama pull phi3:mini` — smaller and faster (~2.3GB)  
   `ollama pull gemma2:2b` — Google's model, very capable for size  

Ollama runs a local HTTP server at `http://localhost:11434`. We talk to it via Python `requests`.

In [ ]:
import requests
import json

OLLAMA_URL = 'http://localhost:11434'

# Check if Ollama is running
try:
    response = requests.get(f'{OLLAMA_URL}/')
    print('Ollama is running!', response.text)
except requests.exceptions.ConnectionError:
    print('ERROR: Ollama is not running.')
    print('Run `ollama serve` in a terminal first.')

In [ ]:
# See what models you have downloaded
response = requests.get(f'{OLLAMA_URL}/api/tags')
models = response.json().get('models', [])

if models:
    print('Downloaded models:')
    for m in models:
        size_gb = m.get('size', 0) / 1e9
        print(f"  - {m['name']}  ({size_gb:.1f} GB)")
else:
    print('No models downloaded yet.')
    print('Run: ollama pull llama3.2:3b')

## Part 1: Simple Chat

The `/api/chat` endpoint accepts a list of messages and returns a response.

In [ ]:
MODEL = 'llama3.2:3b'  # Change this to any model you have downloaded

def chat(messages, model=MODEL):
    """Send a list of messages and return the model's reply text."""
    response = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model': model,
            'messages': messages,
            'stream': False,  # wait for full response
        }
    )
    return response.json()['message']['content']

# Single message
reply = chat([
    {'role': 'user', 'content': 'What is machine learning? Explain in 2 sentences.'}
])
print(reply)

## Part 2: Multi-turn Conversation

To have a conversation, pass the entire history each time.

In [ ]:
# A simple conversation
history = [
    {'role': 'system', 'content': 'You are a friendly tutor explaining AI concepts to a beginner. Be concise.'}
]

def ask(user_message):
    history.append({'role': 'user', 'content': user_message})
    reply = chat(history)
    history.append({'role': 'assistant', 'content': reply})
    print(f'User: {user_message}')
    print(f'AI: {reply}')
    print()

ask('What is a neural network?')
ask('How is that different from regular programming?')
ask('Can you give me a real-world example?')

## Part 3: Streaming Responses

For a real chat feel, stream the response token-by-token.

In [ ]:
import sys

def chat_stream(messages, model=MODEL):
    """Stream a response, printing tokens as they arrive."""
    response = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={'model': model, 'messages': messages, 'stream': True},
        stream=True
    )
    
    full_reply = ''
    for line in response.iter_lines():
        if line:
            chunk = json.loads(line)
            token = chunk.get('message', {}).get('content', '')
            print(token, end='', flush=True)
            full_reply += token
            if chunk.get('done', False):
                break
    print()  # newline at end
    return full_reply

print('AI: ', end='')
reply = chat_stream([
    {'role': 'user', 'content': 'Write a haiku about training neural networks.'}
])

## Part 4: System Prompts — Changing the AI's Personality

In [ ]:
# Try different system prompts to see how it changes behavior
system_prompts = {
    'pirate': 'You are a pirate. Answer every question in pirate speak.',
    'expert': 'You are a terse expert. Give only essential information. No pleasantries.',
    'socratic': 'You are a Socratic teacher. Instead of answering, ask questions that guide the user to discover the answer themselves.',
}

question = 'What is gradient descent?'

for persona, system in system_prompts.items():
    reply = chat([
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': question},
    ])
    print(f'--- {persona.upper()} ---')
    print(reply)
    print()

## Exercises

1. **Change the model**: Try `phi3:mini` or `gemma2:2b` instead of llama3.2 — same code works!
2. **Build a simple chatbot**: Write a `while True` loop that takes input from the user and calls `ask()`
3. **JSON output**: Ask the model to respond only in JSON format for structured data extraction

```python
reply = chat([{
    'role': 'user',
    'content': 'List 3 ML algorithms as JSON: [{"name": ..., "type": ..., "use_case": ...}]. Only output JSON.'
}])
data = json.loads(reply)
print(data)
```

**Next:** `02_simple_rag.ipynb` — give the model your own documents to reason about!